### Retiro Fugas

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text


In [4]:

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'

filename='retiro_tc_20260717_01.xlsx'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_csv, filename)
df = pd.read_excel(ruta_archivo)

df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)

df = df.rename(columns={
    f'{name_dni}': 'NUMERO_DOCUMENTO'
})
df=df[["NUMERO_DOCUMENTO"]]
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	select NUMERO_DOCUMENTO
	from DANTALION.dbo.Base_Maestra_Diners_TC_Vigente
    WHERE RETIRO IS NULL OR RETIRO =''
"""
df_tc = pd.read_sql(query, engine_kishin)


In [2]:
df_tc.shape

(39853, 1)

In [5]:
campana='dinersTc'
query = f"""
    select FECHA as fecha_gestion,DNI as NUMERO_DOCUMENTO,
    PROMOTOR as promotor,
    ESTADO as estado_venta,
    TRAMA_HORA as tramo_venta,
    MONTO as monto_venta,
    DNIEjecutivo as dni_ejecutivo_venta,
    Producto as producto_venta,
    Producto as title,
    Celular as cel_venta,1 as venta,
    subcampana
    from SAMANTHA.dbo.Ventas_Target
    where campana='{campana}'
    and cast(fecha as date) between '{fecha_mes_base}' and EOMONTH('{fecha_mes_base}')
"""
df_ventas = pd.read_sql(query, engine_samantha)

In [6]:
df_ventas.count()

fecha_gestion          201
NUMERO_DOCUMENTO       201
promotor               201
estado_venta           201
tramo_venta            201
monto_venta              0
dni_ejecutivo_venta    201
producto_venta           0
title                    0
cel_venta              201
venta                  201
subcampana               0
dtype: int64

In [7]:
df = df[
    ~df["NUMERO_DOCUMENTO"].isin(df_ventas["NUMERO_DOCUMENTO"])
].copy()


In [8]:
df.merge(df_tc, on='NUMERO_DOCUMENTO', how='inner').count()


NUMERO_DOCUMENTO    1884
dtype: int64

In [9]:

list_dni = (
    df['NUMERO_DOCUMENTO']
    .dropna()
    .drop_duplicates()
    .tolist()
)
in_clause = ",".join(f"'{x}'" for x in list_dni)

in_clause

"'70616397','43850905','72976172','44755864','47481829','70575810','71561978','43756334','45394060','71570358','47607845','70558173','72393032','73378522','70102727','41598938','46833055','43518318','73752170','45158903','48063442','43487571','40242634','70021940','44568166','71478967','72697908','47448366','47491907','44972117','07127441','75459989','46641795','48151999','06267077','45572038','73188653','74212430','42993532','42429324','71868257','75438478','47832656','40293366','47448052','44115936','75520687','44069960','70865637','45999459','75368009','40105761','41677855','44155217','41036230','48152453','73485144','40062769','42977517','73958376','10314489','46546698','72870934','71893746','47463427','47016100','72571782','70326843','48014670','70691861','44519661','45836593','71943121','70125004','40656563','76455747','47901252','77079160','77148412','46197950','45864409','48486912','47509453','47104593','46789013','72132607','47173197','76344843','40375724','46097752','44837228

In [9]:
in_clause = ",".join(f"'{x}'" for x in list_dni)

In [11]:
df.count()

NUMERO_DOCUMENTO    2030
dtype: int64

In [13]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners_TC
            SET RETIRO = 'RETIRO_17'
            WHERE NUMERO_DOCUMENTO IN ({in_clause})
                and fecha_envio >= '{fecha_mes_base}'
                AND fecha_envio <= EOMONTH('{fecha_mes_base}')
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 1884


In [10]:
fecha_mes_base

'2026-07-01'

In [ ]:

df_ventas.head()

,fecha_gestion,NUMERO_DOCUMENTO,promotor,estado_venta,tramo_venta,monto_venta,dni_ejecutivo_venta,producto_venta,title,cel_venta,venta,subcampana
0,2026-07-01,07259135,08728081,1.-VALIDADA,9,None,08728081,None,None,981260227,1,None
1,2026-07-01,07779510,44592143,6.-SIN VALIDAR,13,None,44592143,None,None,989371280,1,None
2,2026-07-01,40102384,44592143,1.-VALIDADA,10,None,44592143,None,None,932595948,1,None
3,2026-07-01,40552162,47202133,1.-VALIDADA,10,None,47202133,None,None,988594366,1,None
4,2026-07-01,41266190,44592143,1.-VALIDADA,9,None,44592143,None,None,940049145,1,None


In [14]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_tc", "SP tNumeros diners TC")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC SA")

SP tNumeros diners TC | realizado | duración: 25.21 seg
SP actualizar diners TC Zeus | realizado | duración: 35.56 seg
SP actualizar diners TC SA | realizado | duración: 3.5 seg


In [8]:
query = f"""
	select NUMERO_DOCUMENTO,PROB_CONTACTO,concat('2026-07-',right(retiro,2)) as retiro
	from DANTALION.dbo.Base_Maestra_Diners_TC_Vigente
    WHERE RETIRO IS not NULL OR RETIRO <>''
"""
df_tc = pd.read_sql(query, engine_kishin)

In [9]:
df_tc.head()

,NUMERO_DOCUMENTO,PROB_CONTACTO,retiro
0,47216152,D,2026-07-01
1,71561197,D,2026-07-01
2,42968074,C,2026-07-01
3,73473064,E,2026-07-01
4,46510882,C,2026-07-01


In [ ]:


server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
fecha_mes_base='2026-06-01'
campana=fecha_a_nombre('2026-05-01')
query = f"""
select dni as NumDoc, 1 as venta_target from SAMANTHA.dbo.Ventas_Target
where CAMPANA='Diners'
and CONVERT(DATE, FECHA) >= CONVERT(DATE, '{fecha_mes_base}')
AND CONVERT(DATE, FECHA) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
   
"""
df_venta = pd.read_sql(query, engine_zeus)

df_target = df_tc.merge(
    df_venta,
    on='NumDoc',
    how='left'
)
df_final = df_target.merge(
    df,
    on='NumDoc',
    how='inner'
)
df_final = (
    df_final[df_final['venta_target'].isnull()]
    .drop(columns=['venta_target'])
)
df_final.rename(
    columns={
        'Importe Solicitado': 'Monto'
    },
    inplace=True
)
print(df_final.columns.tolist())

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


['NumDoc', 'TIPO_PRODUCTO', 'RETIRO', 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Monto', 'fecha', 'hora']


In [72]:
df_final=df_final[df_final['Monto'].notnull()]

In [78]:
df_final[['TIPO_PRODUCTO','Canal', 'Monto', 'fecha']].head()


,TIPO_PRODUCTO,Canal,Monto,fecha
0,PPD,CANALES DIGITALES,41100.0,2026-06-04
1,PPD,CANALES DIGITALES,13800.0,2026-06-04
2,PPD,CONTACT CENTER,6000.0,2026-06-04


In [ ]:

total_monto_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].sum()

total_ope_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].count()

print(f'PPD | Monto total: {int(total_monto_ppd)} |  Total operaciones {int(total_ope_ppd)} ')

PPD | Monto total: 60900 |  Total operaciones 3 


In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'PPD_plus.xlsx')
# df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
list_dni = (
    df_final.loc[df_final['RETIRO'].isna(), 'NumDoc']
    .dropna()
    .drop_duplicates()
    .tolist()
)

in_clause = ",".join(f"'{x}'" for x in list_dni)

try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners
            SET RETIRO = 'RETIRO'
            WHERE NumDoc IN ({in_clause})
                and CONVERT(DATE, fecha_envio) >= CONVERT(DATE, '{fecha_mes_base}')
                AND CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)


In [77]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 5.48 seg
SP actualizar diners Zeus | realizado | duración: 5.54 seg
SP actualizar diners SA | realizado | duración: 1.5 seg
